In [1]:
import pandas as pd
sampled_sentences = pd.read_csv("../01_data/sample_annotations.csv")

In [51]:
print(sampled_sentences["sentence"].iloc[300])

In terms of the number of families who will be affected, higher-rate tax is paid by 15% of taxpayers, and the decision that we have taken is to say that child benefit should not be received by families where there is a higher-rate taxpayer.


In [32]:
import re

def clean_text(text):
    # return empty string if the text column is not a string
    if not isinstance(text, str):
        return ""

    # replace misencoded punctuation
    replacements = {
        "Â£": "£",
        "â€œ": "“",
        "â€": "”",
        "â€˜": "‘",
        "â€™": "’",
        "â€“": "–",
        "â€”": "—",
        "â€¦": "…",
        "â€": '"',
        "Ã©": "é",
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # normalize whitespaces
    text = re.sub(r"\s+", " ", text)

    # remove leading and trailing whitespaces
    return text.strip()

def create_llm_annotations(text, spans):

    # sort the spans according to their start and end index
    spans = sorted(spans, key=lambda x: x["start"])

    # store the text and set index variable
    llm_text = ""
    last_idx = 0

    # loop through spans and add the span with custom characters
    for span in spans:
        llm_text += text[last_idx:span["start"]]
        llm_text += f"@@{text[span["start"]:span["end"]]}##"
        last_idx = span["end"]

    # add the rest of the text
    llm_text += text[last_idx:]

    return llm_text

def llm_output_to_bio(annotated_text):

    # split words via a regex
    words = re.findall(r"@@.*?##|\w+|'\w+|[^\w\s]", annotated_text)

    # empty list to store the bio tags
    bio_tags = []

    # loop through all words
    for word in words:

        # if it is an annotated span, split words and assign bio labels
        if word.startswith("@@") and word.endswith("##"):
            entity_text = word[2:-2]
            entity_words = re.findall(r"\w+|'\w+|[^\w\s]", entity_text)
            for i, t in enumerate(entity_words):
                tag = "B-sg" if i == 0 else "I-sg"
                bio_tags.append((t, tag))
        else:
            # otherwise assign O tag
            bio_tags.append((word, "O"))

    return bio_tags

def tokenize_word_level(sentence):
    
    # get all words' start and end index
    word_spans = []
    char_idx = 0

    # split sentence via regex which ensures to also split at punctuation
    words = re.findall(r"\w+|'\w+|[^\w\s]", sentence)

    for word in words:
        start_idx = sentence.find(word, char_idx)
        end_idx = start_idx + len(word)
        word_spans.append((start_idx, end_idx))
        char_idx = end_idx
    
    return words, word_spans

In [36]:
sentence = "The Energy Bill“sends a strong signal to investors”."
annotations = [{'start': 41,
               'end': 50,
               'text': 'investors',
               'tag': 'sg_neutral'}]
annotated_text = create_llm_annotations(sentence, annotations)
llm_output_to_bio(annotated_text)


[('The', 'O'),
 ('Energy', 'O'),
 ('Bill', 'O'),
 ('“', 'O'),
 ('sends', 'O'),
 ('a', 'O'),
 ('strong', 'O'),
 ('signal', 'O'),
 ('to', 'O'),
 ('investors', 'B-sg'),
 ('”', 'O'),
 ('.', 'O')]

In [8]:
import pandas as pd

parl_questions_df = pd.read_csv("../01_data/parliamentary_questions_df.csv")
parl_questions_df.head()

,date,agenda,speechnumber,speaker,party,party.facts.id,chair,terms,text,parliament,iso3country,month,year,clean_text,sentences
0,2010-06-02,Departmental Expenditure [Oral Answers to Ques...,1,Stephen Metcalfe,Con,1567.0,False,16,What mechanism he plans to use to review the v...,UK-HouseOfCommons,GBR,6,2010,What mechanism he plans to use to review the v...,"[""What mechanism he plans to use to review the..."
1,2010-06-02,Departmental Expenditure [Oral Answers to Ques...,2,Andrew Mitchell,Con,1567.0,False,55,We will fundamentally change the way in which ...,UK-HouseOfCommons,GBR,6,2010,We will fundamentally change the way in which ...,['We will fundamentally change the way in whic...
2,2010-06-02,Departmental Expenditure [Oral Answers to Ques...,3,Stephen Metcalfe,Con,1567.0,False,86,May I take this opportunity to welcome my righ...,UK-HouseOfCommons,GBR,6,2010,May I take this opportunity to welcome my righ...,['May I take this opportunity to welcome my ri...
3,2010-06-02,Departmental Expenditure [Oral Answers to Ques...,4,Andrew Mitchell,Con,1567.0,False,114,I thank my hon. Friend for his kind remarks. A...,UK-HouseOfCommons,GBR,6,2010,I thank my hon. Friend for his kind remarks. A...,['I thank my hon. Friend for his kind remarks....
4,2010-06-02,Departmental Expenditure [Oral Answers to Ques...,5,Douglas Alexander,Lab,1516.0,False,101,"With your permission, Mr Speaker, let me retur...",UK-HouseOfCommons,GBR,6,2010,"With your permission, Mr Speaker, let me retur...","['With your permission, Mr Speaker, let me ret..."


In [9]:
import ast
parl_questions_df['sentences'] = parl_questions_df['sentences'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
sentences_df = parl_questions_df.explode('sentences').reset_index(drop=True)
sentences_df = sentences_df.rename(columns={'sentences': 'sentence'})
sentences_df.head()

,date,agenda,speechnumber,speaker,party,party.facts.id,chair,terms,text,parliament,iso3country,month,year,clean_text,sentence
0,2010-06-02,Departmental Expenditure [Oral Answers to Ques...,1,Stephen Metcalfe,Con,1567.0,False,16,What mechanism he plans to use to review the v...,UK-HouseOfCommons,GBR,6,2010,What mechanism he plans to use to review the v...,What mechanism he plans to use to review the v...
1,2010-06-02,Departmental Expenditure [Oral Answers to Ques...,2,Andrew Mitchell,Con,1567.0,False,55,We will fundamentally change the way in which ...,UK-HouseOfCommons,GBR,6,2010,We will fundamentally change the way in which ...,We will fundamentally change the way in which ...
2,2010-06-02,Departmental Expenditure [Oral Answers to Ques...,2,Andrew Mitchell,Con,1567.0,False,55,We will fundamentally change the way in which ...,UK-HouseOfCommons,GBR,6,2010,We will fundamentally change the way in which ...,We will gain maximum value for money for every...
3,2010-06-02,Departmental Expenditure [Oral Answers to Ques...,3,Stephen Metcalfe,Con,1567.0,False,86,May I take this opportunity to welcome my righ...,UK-HouseOfCommons,GBR,6,2010,May I take this opportunity to welcome my righ...,May I take this opportunity to welcome my righ...
4,2010-06-02,Departmental Expenditure [Oral Answers to Ques...,3,Stephen Metcalfe,Con,1567.0,False,86,May I take this opportunity to welcome my righ...,UK-HouseOfCommons,GBR,6,2010,May I take this opportunity to welcome my righ...,I am sure that Members on both sides of the Ho...


In [11]:
sentences_per_party = sentences_df.groupby("party").size()
import re
regex_pattern = re.compile(r".*\?$")
question_sentences = sentences_df[sentences_df["sentence"].str.match(regex_pattern)]
unique_speeches_df = question_sentences.drop_duplicates(subset=["date", 'speaker', 'speechnumber'])
all_question_sentences = pd.merge(sentences_df, unique_speeches_df[["date", "speaker", "speechnumber"]], on=["date", "speaker", "speechnumber"], how="inner")

514236

In [37]:
print(len(sentences_df))
print(len(question_sentences))
print(len(all_question_sentences))

514236
91103
200202


In [35]:
sentences_per_party = sentences_df.groupby("party").size()
sentences_per_party

party
APNI                            222
Birkenhead Social Justice         7
Change UK                        70
Con                          349331
DUP                            3362
GPEW                            531
Independent                     525
Lab                          110446
LibDem                        33412
PlaidCymru                     1410
Respect                           9
SDLP                            776
SNP                           13609
The Independents                 12
UKIP                            144
UUP                             243
dtype: int64

In [34]:
questions_per_party = all_question_sentences.groupby("party").size()
questions_per_party

party
APNI                           157
Birkenhead Social Justice        6
Change UK                       57
Con                          76825
DUP                           3137
GPEW                           416
Independent                    445
Lab                          95204
LibDem                        9933
PlaidCymru                    1244
Respect                          9
SDLP                           696
SNP                          11704
The Independents                11
UKIP                           122
UUP                            209
dtype: int64

In [39]:
# print some example questions
for idx in range(20):
    print(f"Question from party {question_sentences["party"].iloc[idx]}")
    #print("-"*100)
    print(question_sentences["sentence"].iloc[idx])
    print("-"*100)

Question from party Con
May I take this opportunity to welcome my right hon. Friend to the Dispatch Box and to congratulate him on his new and important role?
----------------------------------------------------------------------------------------------------
Question from party Con
Will he reassure the House and my constituents that value for money will be at the heart of his Department's vital work in tackling poverty in the poorest countries in the world?
----------------------------------------------------------------------------------------------------
Question from party Lab
May I ask whether he regards educating young girls in Afghanistan as a valuable part of that comprehensive approach or whether he agrees with the Defence Secretary that it is simply“education policy in a broken 13th-century country”?
----------------------------------------------------------------------------------------------------
Question from party Lab
Has he also secured the re-education of the new Secre